# TinySLM C2 — piloto de rotulagem por modelo (v02)

Este notebook **não treina nem altera o checkpoint C2**. Descarrega automaticamente o dataset v02 do GitHub e executa um ensaio de **12 perguntas, 3 trabalhadores**, usando um modelo professor via OpenAI API. Os rótulos gerados são propostas: não substituem validação humana.

O histórico do projeto já utilizava lotes de pedidos e respostas estruturadas (`distill.py`). Este ensaio reutiliza o mesmo princípio, com execução paralela e gravação incremental em JSONL.

**Antes de executar com API:** no Kaggle, em *Add-ons → Secrets*, criar o segredo `OPENAI_API_KEY`, ativar o acesso do notebook e confirmar que Internet está ligada. A subscrição ChatGPT/Codex não equivale a crédito de API; as chamadas podem ter custos.


In [ ]:
%pip -q install "openai>=1.75,<3"

In [ ]:
from pathlib import Path
import urllib.request, zipfile, io

ROOT = Path('/kaggle/working/tinyslm_intents_teacher_pilot')
ROOT.mkdir(parents=True, exist_ok=True)
REPO_RAW = 'https://raw.githubusercontent.com/smartlearningci/tinyLLM/main/'
ZIP_URL = REPO_RAW + 'data/intent_candidates/TinySLM_C2_intents_candidates_v02.zip'
SCRIPT_URL = REPO_RAW + 'tools/recipe_intents/label_intents_teacher.py'

with urllib.request.urlopen(ZIP_URL, timeout=120) as response:
    archive = response.read()
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    assert z.testzip() is None
    assert 'intent_dataset_v02/review_sample_v02.csv' in z.namelist()
    z.extract('intent_dataset_v02/review_sample_v02.csv', ROOT)

SCRIPT = ROOT / 'label_intents_teacher.py'
urllib.request.urlretrieve(SCRIPT_URL, SCRIPT)
SAMPLE = ROOT / 'intent_dataset_v02/review_sample_v02.csv'
print('Dataset:', SAMPLE, '| size:', SAMPLE.stat().st_size, 'bytes')
print('Script:', SCRIPT)

## 1. Simular a seleção — zero chamadas pagas

Executar primeiro para ver as 12 perguntas e os rótulos v02. O modelo professor ainda não intervém.


In [ ]:
import subprocess, sys

common = [sys.executable, str(SCRIPT), '--input', str(SAMPLE),
          '--output', str(ROOT / 'pilot_teacher_labels.jsonl'),
          '--limit', '12', '--workers', '3', '--seed', '42']
subprocess.run(common + ['--dry-run'], check=True)

## 2. Rotular via OpenAI API (opcional, sujeito a custo)

Por segurança, esta célula **não envia pedidos** até alterares `EXECUTAR_API` para `True`. O segredo é lido do cofre do Kaggle, não deve ser escrito no notebook. Escolhe um modelo disponível na tua conta; o predefinido é `gpt-4.1-mini`.


In [ ]:
EXECUTAR_API = False
MODELO_PROFESSOR = 'gpt-4.1-mini'

if EXECUTAR_API:
    import os
    from kaggle_secrets import UserSecretsClient
    os.environ['OPENAI_API_KEY'] = UserSecretsClient().get_secret('OPENAI_API_KEY')
    subprocess.run(common + ['--model', MODELO_PROFESSOR], check=True)
else:
    print('Nenhuma chamada API realizada. Altera EXECUTAR_API=True para iniciar.')

## 3. Comparar rótulos e descarregar resultados

A coluna `human_verified` fica sempre `false`. O consenso entre dois modelos **não é verdade de referência**. O ficheiro permanece em Output mesmo se algumas chamadas falharem; ao voltar a executar, casos concluídos não são repetidos.


In [ ]:
import json, collections

RESULT = ROOT / 'pilot_teacher_labels.jsonl'
if RESULT.exists():
    rows = [json.loads(x) for x in RESULT.read_text(encoding='utf-8').splitlines() if x.strip()]
    print('Linhas:', len(rows))
    print('Estados:', dict(collections.Counter(x['status'] for x in rows)))
    print('Concordância com candidatos:', dict(collections.Counter(str(x.get('agrees_with_candidate', 'n/a')) for x in rows)))
    for x in rows[:12]:
        print(x['question'][:75], '\n  V02:', x['candidate'], '\n  Teacher:', x.get('teacher'), '\n')
else:
    print('Ainda não há resultados; faz o piloto com a API quando estiveres pronto.')